In [1]:
from peft import LoraConfig, get_peft_model

c:\Users\weich\anaconda3\envs\LLM\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "../model/huggingface_transformer_format"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:07<00:00, 20.80it/s, Materializing param=model.norm.weight]                              
Some parameters are on the meta device because they were offloaded to the disk and cpu.


In [4]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


In [5]:
# Load dataset
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="../data/train.json"
)

Generating train split: 2 examples [00:00, 41.77 examples/s]


In [6]:
# Check Structure
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response'],
        num_rows: 2
    })
})
{'prompt': 'Explain machine learning in simple terms.', 'response': 'Machine learning is when computers learn patterns from data instead of being explicitly programmed.'}


In [7]:
def format_prompt(example):
    return f"""### Prompt:
{example['prompt']}

### Response:
{example['response']}"""

In [9]:
def tokenize(example):

    text = format_prompt(example)

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


dataset = dataset.map(tokenize)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]


ValueError: Asking to pad but the tokenizer does not have a padding token. Please select a token to use as `pad_token` `(tokenizer.pad_token = tokenizer.eos_token e.g.)` or add a new pad token via `tokenizer.add_special_tokens({'pad_token': '[PAD]'})`.